# Семинар 3 — функции потерь и градиентный спуск

Сегодня соберём один прозрачный механизм обучения и несколько раз переиспользуем его:

$$
\text{целевая функция}
\longrightarrow
\text{градиент}
\longrightarrow
\text{шаг}
\longrightarrow
\text{новые параметры}.
$$

Маршрут занятия:

1. несколько шагов градиентного спуска руками;
2. одна общая функция `gradient_descent`;
3. тот же алгоритм для вектора параметров;
4. линейная регрессия с MSE и сравнение с `lstsq`;
5. те же модель и оптимизатор, но функция Хубера на данных с крупными ошибками;
6. готовая демонстрация влияния масштаба признаков.

Полноценный класс `LinearRegressionGD` здесь не строим: это отдельная инженерная задача для домашней работы.

## Техническая рамка

В этом семинаре данные **синтетические** и генерируются прямо в ноутбуке с фиксированными начальными значениями генератора случайных чисел.

Основной набор данных будет построен по схеме

$$
x\sim U(0,5),
\qquad
Y=10+3x+\varepsilon,
\qquad
\varepsilon\sim\mathcal N(0,1.2^2).
$$

Позже мы создадим копию тех же данных и намеренно испортим три значения целевой переменной. Это контролируемый эксперимент: мы заранее знаем, какие наблюдения были изменены и зачем.

Внешние файлы и сеть не нужны.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
SEED = 42

## 1. Настройка: один шаг градиентного спуска

Возьмём функцию

$$
f(w)=(w-3)^2,
\qquad
f'(w)=2(w-3).
$$

Минимум находится в $w=3$, но алгоритму эту точку заранее не сообщаем.

Правило градиентного спуска в одной переменной:

$$
w_{t+1}=w_t-\eta f'(w_t).
$$

Пусть

$$
w_0=0,
\qquad
\eta=0.1.
$$

Первый шаг:

$$
f'(0)=-6,
\qquad
w_1=0-0.1(-6)=0.6.
$$

Второй шаг:

$$
f'(0.6)=-4.8,
\qquad
w_2=0.6-0.1(-4.8)=1.08.
$$

Перед запуском кода: **какой знак будет у производной на следующих шагах, пока $w<3$? В какую сторону будет двигаться $w$?**

После ручного расчёта сразу посмотрим на те же шаги на графике функции.

In [ ]:
w = 0.0
learning_rate = 0.1
w_manual = [w]

for step in range(5):
    value = (w - 3.0) ** 2
    grad = 2.0 * (w - 3.0)
    print(f"шаг {step}: w={w:7.4f}, f(w)={value:8.4f}, f'(w)={grad:8.4f}")
    w = w - learning_rate * grad
    w_manual.append(w)

w_grid = np.linspace(-0.5, 4.5, 300)
f_grid = (w_grid - 3.0) ** 2
f_manual = (np.asarray(w_manual) - 3.0) ** 2

plt.figure(figsize=(7.4, 4.2))
plt.plot(w_grid, f_grid, linewidth=2, label=r"$f(w)=(w-3)^2$")
plt.plot(w_manual, f_manual, marker="o", linewidth=1.6, label="первые шаги GD")
for step, (w_step, f_step) in enumerate(zip(w_manual, f_manual)):
    plt.annotate(str(step), (w_step, f_step), xytext=(5, 5), textcoords="offset points")
plt.scatter([3.0], [0.0], marker="*", s=110, label="минимум")
plt.xlabel("w")
plt.ylabel("f(w)")
plt.title("Первые шаги градиентного спуска на параболе")
plt.legend()
plt.grid(alpha=0.22)
plt.show()

## 2. Реализуем один общий градиентный спуск

Для вектора параметров правило из лекции имеет вид

$$
\boxed{
\theta^{(t+1)}
=
\theta^{(t)}
-
\eta\nabla f\bigl(\theta^{(t)}\bigr)
}.
$$

В нашей функции это будет означать буквально пять действий:

1. начать с `theta0`;
2. вычислить градиент целевой функции в текущей точке;
3. обновить `theta = theta - learning_rate * grad`;
4. сохранить новые параметры и новое значение целевой функции;
5. повторить обновление `n_steps` раз.

Функции `objective_fn` и `grad_fn` передаются в оптимизатор как аргументы. Поэтому сам алгоритм движения можно написать один раз, а затем менять конкретную целевую функцию и её градиент.

Позже целевая функция будет зависеть не только от параметров, но и от фиксированных данных, например `X` и `y`. Чтобы эта зависимость была видна прямо в месте вызова, функция принимает кортеж `fixed_args`. Например,

```python
fixed_args=(X, y)
```

означает, что внутри оптимизатора вызов

```python
objective_fn(theta, *fixed_args)
```

эквивалентен вызову

```python
objective_fn(theta, X, y)
```

Синтаксис `*fixed_args` здесь просто распаковывает элементы кортежа в отдельные аргументы функции.

### Контракт функции

Ниже используется подробная строка документации (`docstring`). Она описывает смысл аргументов, ожидаемые размеры массивов и возвращаемые значения. Такой контракт полезен даже для небольшой учебной функции: по нему можно понять, как её использовать, не читая всё тело реализации.

Перед циклом есть техническая строка

```python
theta = np.array(theta0, dtype=float, copy=True)
```

Она делает три вещи:

- превращает `theta0` в NumPy-массив;
- переводит значения в `float`, потому что шаги градиентного спуска обычно дробные;
- создаёт независимую копию, чтобы функция не меняла исходный объект `theta0` снаружи.

In [ ]:
def gradient_descent(
    theta0,
    objective_fn,
    grad_fn,
    learning_rate,
    n_steps,
    fixed_args=(),
):
    """
    Минимизирует целевую функцию градиентным спуском с постоянным шагом.

    Parameters
    ----------
    theta0 : array-like, shape (d,)
        Начальное значение вектора из d параметров.
    objective_fn : callable
        Целевая функция. Первый аргумент — theta; после него при
        необходимости передаются элементы fixed_args.
    grad_fn : callable
        Функция градиента. Возвращает одномерный массив длины d:
        по одной частной производной на каждый параметр theta.
    learning_rate : float
        Положительный темп обучения eta.
    n_steps : int
        Число обновлений параметров.
    fixed_args : tuple, default=()
        Дополнительные неизменяемые аргументы для objective_fn и grad_fn.
        Например, для линейной регрессии это может быть (X, y).

    Returns
    -------
    theta : np.ndarray, shape (d,)
        Вектор параметров после последнего обновления.
    history : dict[str, np.ndarray]
        История параметров и значений целевой функции.
        history["theta"] имеет размер (n_steps + 1, d),
        history["objective"] — длину n_steps + 1.
    """
    theta = np.array(theta0, dtype=float, copy=True)
    history = {
        "theta": [theta.copy()],
        "objective": [float(objective_fn(theta, *fixed_args))],
    }

    for _ in range(n_steps):
        # Инфраструктура fixed_args уже готова:
        # grad_fn(theta, *fixed_args) передаёт theta и все фиксированные аргументы.
        grad = np.array(grad_fn(theta, *fixed_args), dtype=float)

        # TODO 1: сделайте шаг по антиградиенту.
        theta = ...

        # TODO 2: сохраните обновлённые параметры.
        history["theta"].append(...)

        # TODO 3: сохраните новое значение целевой функции.
        history["objective"].append(...)

    history["theta"] = np.asarray(history["theta"])
    history["objective"] = np.asarray(history["objective"])
    return theta, history

### Быстрая проверка через `assert`

Конструкция

```python
assert условие
```

работает как маленький автоматический тест. Если условие истинно, выполнение продолжается. Если оно ложно, Python останавливается с `AssertionError`.

В этом ноутбуке готовые `assert` помогают быстро проверить реализацию. Писать собственную систему тестов сегодня не требуется.

In [ ]:
assert 2 + 2 == 4
print("Пример assert пройден.")

In [ ]:
def parabola_func(theta):
    """Возвращает f(w) = (w - 3)^2 для theta = [w]."""
    w = theta[0]
    return (w - 3.0) ** 2


def parabola_grad(theta):
    """Возвращает массив из одной производной df/dw."""
    w = theta[0]
    return np.array([2.0 * (w - 3.0)])


theta_final, parabola_history = gradient_descent(
    theta0=np.array([0.0]),
    objective_fn=parabola_func,
    grad_fn=parabola_grad,
    learning_rate=0.1,
    n_steps=40,
)

assert theta_final.shape == (1,)
assert parabola_history["theta"].shape == (41, 1)
assert parabola_history["objective"].shape == (41,)
assert np.isfinite(theta_final).all()
assert abs(theta_final[0] - 3.0) < 1e-3
assert parabola_history["objective"][-1] < parabola_history["objective"][0]

print("Все проверки gradient_descent пройдены.")
print("Финальное w:", theta_final[0])
print("Финальное f(w):", parabola_history["objective"][-1])

### Как влияет темп обучения

Для той же параболы сравним три значения:

$$
\eta=0.05,
\qquad
\eta=0.8,
\qquad
\eta=1.1.
$$

**До запуска** попробуйте распределить их по трём описаниям:

- сходится без смены стороны относительно минимума;
- перескакивает через минимум, но амплитуда уменьшается;
- расходится.

In [ ]:
learning_rates = [0.05, 0.8, 1.1]

fig, axes = plt.subplots(2, 3, figsize=(13.2, 7.2))

for col, eta in enumerate(learning_rates):
    theta_eta, history_eta = gradient_descent(
        theta0=np.array([0.0]),
        objective_fn=parabola_func,
        grad_fn=parabola_grad,
        learning_rate=eta,
        n_steps=14,
    )
    w_path = history_eta["theta"][:, 0]

    # Верхний ряд: первые шаги непосредственно на параболе.
    n_show = min(7, len(w_path))
    w_visible = w_path[:n_show]
    w_min = min(-0.5, w_visible.min() - 0.8)
    w_max = max(4.5, w_visible.max() + 0.8)
    grid = np.linspace(w_min, w_max, 350)

    ax_top = axes[0, col]
    ax_top.plot(grid, (grid - 3.0) ** 2, linewidth=2)
    ax_top.plot(
        w_visible,
        (w_visible - 3.0) ** 2,
        marker="o",
        linewidth=1.4,
        markersize=4,
    )
    ax_top.scatter([3.0], [0.0], marker="*", s=85)
    ax_top.set_title(f"η = {eta}")
    ax_top.set_xlabel("w")
    ax_top.set_ylabel("f(w)")
    ax_top.grid(alpha=0.22)

    # Нижний ряд: значение параметра по итерациям.
    ax_bottom = axes[1, col]
    ax_bottom.plot(np.arange(len(w_path)), w_path, marker="o", markersize=3)
    ax_bottom.axhline(3.0, linestyle="--", linewidth=1)
    ax_bottom.set_xlabel("Номер состояния")
    ax_bottom.set_ylabel("w")
    ax_bottom.grid(alpha=0.25)

fig.suptitle("Одна парабола, разные темпы обучения", y=1.01)
plt.tight_layout()
plt.show()

## 3. Тот же алгоритм для вектора параметров

Теперь параметр состоит из двух координат:

$$
\theta=
\begin{pmatrix}
w_1\\
w_2
\end{pmatrix}.
$$

Возьмём функцию

$$
f(w_1,w_2)
=
(w_1-0.6)^2
+
3(w_2+0.4)^2.
$$

Её градиент:

$$
\nabla f(w_1,w_2)
=
\begin{pmatrix}
2(w_1-0.6)\\
6(w_2+0.4)
\end{pmatrix}.
$$

Задача этого блока простая: убедиться, что **сам `gradient_descent` менять не нужно**. Меняется только функция, которая вычисляет градиент.

In [ ]:
def quadratic_func(theta):
    """Квадратичная функция двух переменных из примера выше."""
    w1, w2 = theta
    return (w1 - 0.6) ** 2 + 3.0 * (w2 + 0.4) ** 2

In [ ]:
def quadratic_gradient(theta):
    """Возвращает две частные производные: по w1 и по w2."""
    w1, w2 = theta

    # TODO: соберите две частные производные в один NumPy-массив.
    return ...

In [ ]:
assert np.allclose(quadratic_gradient(np.array([0.6, -0.4])), [0.0, 0.0])
assert np.allclose(quadratic_gradient(np.array([1.6, 0.6])), [2.0, 6.0])
print("Проверки quadratic_gradient пройдены.")

In [ ]:
theta_2d, history_2d = gradient_descent(
    theta0=np.array([-2.0, 2.0]),
    objective_fn=quadratic_func,
    grad_fn=quadratic_gradient,
    learning_rate=0.12,
    n_steps=30,
)

print("Финальные параметры:", theta_2d)
print("Финальное значение функции:", history_2d["objective"][-1])

In [ ]:
w1_grid = np.linspace(-2.5, 2.5, 220)
w2_grid = np.linspace(-1.8, 2.5, 220)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z = (W1 - 0.6) ** 2 + 3.0 * (W2 + 0.4) ** 2
path = history_2d["theta"]

plt.figure(figsize=(7.2, 5.2))
plt.contour(W1, W2, Z, levels=18)
plt.plot(path[:, 0], path[:, 1], marker="o", markersize=3, linewidth=1.5)
plt.scatter([0.6], [-0.4], s=70, marker="*", label="минимум")
plt.xlabel("w1")
plt.ylabel("w2")
plt.title("Траектория того же gradient_descent в двух измерениях")
plt.legend()
plt.grid(alpha=0.15)
plt.show()

## 4. Линейная регрессия через тот же градиентный спуск

Теперь переходим к модели, которую уже изучали раньше. Сгенерируем данные

$$
Y=10+3x+\varepsilon,
\qquad
\varepsilon\sim\mathcal N(0,1.2^2).
$$

In [ ]:
rng = np.random.default_rng(SEED)
n = 60
x = np.sort(rng.uniform(0.0, 5.0, size=n))
noise = rng.normal(0.0, 1.2, size=n)
y = 10.0 + 3.0 * x + noise

plt.figure(figsize=(7.2, 4.2))
plt.scatter(x, y, s=30)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Синтетические данные для линейной регрессии")
plt.grid(alpha=0.2)
plt.show()

print("Диапазон y:", (y.min(), y.max()))

### Короткая памятка по матричным операциям

Добавим столбец единиц и будем работать с

$$
X\in\mathbb R^{n\times q},
\qquad
\beta\in\mathbb R^q,
\qquad
y\in\mathbb R^n.
$$

В нашем примере $q=2$: свободный член и один числовой признак.

Полезные формы:

$$
X\beta\in\mathbb R^n,
\qquad
r=y-X\beta\in\mathbb R^n,
\qquad
X^\top r\in\mathbb R^q.
$$

NumPy-запись:

```python
X @ beta          # матрица на вектор
X.T @ residuals   # транспонированная матрица на вектор
a * b             # поэлементное умножение
np.linalg.norm(v) # евклидова норма вектора
```

In [ ]:
X = np.column_stack([np.ones_like(x), x])

print("X.shape   =", X.shape)
print("y.shape   =", y.shape)
print("beta имеет размер (2,)")
print("\nПервые 8 строк матрицы X:")
print(X[:8])

Прогноз и MSE мы уже реализовывали на прошлом семинаре, поэтому здесь они **даны готовыми**. Новая работа начинается с градиента MSE.

In [ ]:
def linear_predict(beta, X):
    """
    Строит прогнозы линейной модели X @ beta.

    Parameters
    ----------
    beta : array-like, shape (q,)
        Коэффициенты линейной модели.
    X : np.ndarray, shape (n, q)
        Матрица признаков, уже содержащая столбец единиц.

    Returns
    -------
    np.ndarray, shape (n,)
        Прогнозы для всех объектов.
    """
    return X @ beta


def mse_loss(beta, X, y):
    """Возвращает средний квадрат остатка для линейной модели."""
    residuals = y - linear_predict(beta, X)
    return np.mean(residuals ** 2)

In [ ]:
beta_demo = np.array([10.0, 3.0])
assert linear_predict(beta_demo, X).shape == y.shape
assert np.isscalar(mse_loss(beta_demo, X, y))
print("Готовые linear_predict и mse_loss работают.")

### Реализуем градиент MSE

Из лекции:

$$
\boxed{
\nabla_\beta R(\beta)
=
-\frac{2}{n}
X^\top(y-X\beta)
}.
$$

Перед реализацией полезно проследить размеры промежуточных выражений:

- `y - X @ beta` — вектор из $n$ остатков, размер `(n,)`;
- `X.T @ residuals` — вектор из $q$ компонент, размер `(q,)`;
- итоговый градиент содержит по одной частной производной на каждый коэффициент `beta`.

In [ ]:
def mse_gradient(beta, X, y):
    """
    Вычисляет градиент MSE по коэффициентам линейной модели.

    Parameters
    ----------
    beta : array-like, shape (q,)
        Текущие коэффициенты.
    X : np.ndarray, shape (n, q)
        Матрица признаков со столбцом единиц.
    y : np.ndarray, shape (n,)
        Наблюдаемые значения целевой переменной.

    Returns
    -------
    np.ndarray, shape (q,)
        По одной частной производной MSE на каждый коэффициент beta.
    """
    beta = np.array(beta, dtype=float)

    # TODO: вычислите residuals, затем примените формулу из markdown выше.
    return ...

In [ ]:
# Небольшая проверка на двух объектах, где результат легко посчитать вручную.
X_check = np.array([
    [1.0, 0.0],
    [1.0, 1.0],
])
y_check = np.array([1.0, 3.0])
beta_check = np.array([0.0, 0.0])

# residuals = [1, 3], поэтому X.T @ residuals = [4, 3]
# и -(2 / 2) * [4, 3] = [-4, -3].
assert np.allclose(
    mse_gradient(beta_check, X_check, y_check),
    [-4.0, -3.0],
)
print("Проверка mse_gradient пройдена.")

### Обучаем линейную модель

Теперь в вызове оптимизатора явно передадим фиксированные данные:

```python
fixed_args=(X, y)
```

Внутри `gradient_descent` это превращает вызов `objective_fn(theta, *fixed_args)` в `mse_loss(beta, X, y)`, а вызов градиента — в `mse_gradient(beta, X, y)`.

Оптимизатор меняет только `beta`; матрица `X` и вектор `y` остаются неизменными на всех шагах.

In [ ]:
beta_gd, linear_history = gradient_descent(
    theta0=np.zeros(2),
    objective_fn=mse_loss,
    grad_fn=mse_gradient,
    learning_rate=0.05,
    n_steps=800,
    fixed_args=(X, y),
)

print("Коэффициенты GD:", beta_gd)
print("Финальный MSE:", linear_history["objective"][-1])

In [ ]:
selected_steps = [0, 1, 5, 20, 100, 800]
xx = np.linspace(x.min(), x.max(), 200)
XX = np.column_stack([np.ones_like(xx), xx])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
axes[0].scatter(x, y, s=26, alpha=0.75, label="данные")
for step in selected_steps:
    beta_step = linear_history["theta"][step]
    axes[0].plot(xx, XX @ beta_step, linewidth=1.2, label=f"шаг {step}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Как меняется прямая")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

axes[1].plot(linear_history["objective"])
axes[1].set_xlabel("Номер шага")
axes[1].set_ylabel("MSE")
axes[1].set_title("Целевая функция")
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Сравниваем с `lstsq`

На прошлой неделе `np.linalg.lstsq` решал ту же задачу наименьших квадратов напрямую. Сейчас сравним его коэффициенты с результатом итеративного метода.

Число

$$
\|\beta_{\mathrm{GD}}-\beta_{\mathrm{lstsq}}\|_2
$$

— евклидово расстояние между двумя векторами коэффициентов. В коде его сразу считает `np.linalg.norm`; вручную сумму квадратов вычислять не нужно.

In [ ]:
beta_lstsq = np.linalg.lstsq(X, y, rcond=None)[0]
beta_distance = np.linalg.norm(beta_gd - beta_lstsq)

comparison = pd.DataFrame(
    {
        "GD": beta_gd,
        "lstsq": beta_lstsq,
        "разность": beta_gd - beta_lstsq,
    },
    index=["свободный член", "коэффициент при x"],
)

display(comparison)
print("Евклидово расстояние между коэффициентами:", beta_distance)

assert beta_distance < 1e-4
print("GD пришёл практически к тому же решению, что и lstsq.")

In [ ]:
plt.figure(figsize=(7.4, 4.3))
plt.scatter(x, y, s=28, alpha=0.75, label="данные")
plt.plot(xx, XX @ beta_gd, linewidth=2.2, label="GD")
plt.plot(xx, XX @ beta_lstsq, linewidth=2.0, linestyle="--", label="lstsq")
plt.xlabel("x")
plt.ylabel("y")
plt.title("GD и lstsq решают одну задачу MSE")
plt.legend()
plt.grid(alpha=0.22)
plt.show()

> **Вопрос аудитории:** это две разные модели или два способа решить одну задачу?
>
> **Ответ:** модель одна и та же — линейная, и целевая функция одна и та же — MSE. `lstsq` и градиентный спуск отличаются способом поиска коэффициентов.

## 5. Меняем функцию потерь: MSE и функция Хубера на загрязнённых данных

Сделаем копию **тех же** данных и намеренно увеличим три значения $y$ у объектов с большими $x$. Такой выбор заметно меняет не только положение, но и наклон MSE-прямой, поэтому различие функций потерь хорошо видно на графике.

Это всё ещё контролируемый учебный эксперимент: мы сами знаем, какие наблюдения изменили.

До запуска сформулируйте прогноз: **какая прямая сильнее повернётся к этим трём наблюдениям — обученная по MSE или по функции Хубера?**

In [ ]:
y_contaminated = y.copy()
outlier_idx = np.array([48, 54, 59])
y_contaminated[outlier_idx] += np.array([25.0, 32.0, 38.0])

plt.figure(figsize=(7.2, 4.2))
plt.scatter(x, y_contaminated, s=30, label="загрязнённые данные")
plt.scatter(
    x[outlier_idx],
    y_contaminated[outlier_idx],
    s=85,
    marker="x",
    linewidths=2.0,
    label="искусственно изменённые точки",
)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Три крупных отклонения при больших x")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
beta_mse_contaminated, mse_contaminated_history = gradient_descent(
    theta0=np.zeros(2),
    objective_fn=mse_loss,
    grad_fn=mse_gradient,
    learning_rate=0.05,
    n_steps=800,
    fixed_args=(X, y_contaminated),
)

print("Коэффициенты после обучения по MSE:", beta_mse_contaminated)

### Новая NumPy-функция: `np.clip`

Для функции Хубера сначала вспомним производную по остатку:

$$
L_\delta'(r)
=
\begin{cases}
-\delta, & r<-\delta,\\
r, & |r|\le\delta,\\
\delta, & r>\delta.
\end{cases}
$$

То есть внутри интервала $[-\delta,\delta]$ производная равна самому остатку, а за его границами значение «обрезается» до $-\delta$ или $\delta$.

Именно это поэлементно делает

```python
np.clip(values, lower, upper)
```

- всё, что меньше `lower`, заменяется на `lower`;
- всё, что больше `upper`, заменяется на `upper`;
- значения внутри интервала не меняются.

Поэтому для массива остатков можно компактно написать

```python
np.clip(residuals, -delta, delta)
```

и получить значения $L_\delta'(r_i)$ сразу для всех объектов.

Перед запуском попробуйте устно предсказать результат для массива `[-5, -1, 0, 1.5, 7]` и границ `[-2, 2]`.

In [ ]:
clip_demo = np.array([-5.0, -1.0, 0.0, 1.5, 7.0])
np.clip(clip_demo, -2.0, 2.0)

> **Ответ:** получится `[-2., -1., 0., 1.5, 2.]`. Именно такое «ограничение наклона» происходит у функции Хубера в хвостах.

### Функция Хубера

Для остатка $r=y-\hat y$:

$$
L_\delta(r)=
\begin{cases}
\frac12r^2, & |r|\le\delta,\\[3pt]
\delta\left(|r|-\frac12\delta\right), & |r|>\delta.
\end{cases}
$$

Из предыдущего блока уже видно, почему её производную можно записать как

$$
L_\delta'(r)
=
\operatorname{clip}(r,-\delta,\delta).
$$

Для линейной модели по правилу цепочки получаем

$$
\boxed{
\nabla_\beta R_{\mathrm{Huber}}(\beta)
=
-\frac1n
X^\top
\operatorname{clip}(y-X\beta,-\delta,\delta)
}.
$$

`huber_loss` ниже уже готова. Для **одного** остатка кусочную формулу можно было бы реализовать обычным `if`. Но у нас сразу целый массив остатков, поэтому удобно использовать `np.where(condition, a, b)`: он применяет ту же развилку поэлементно ко всему массиву.

Нужно реализовать только градиент. **Сам `gradient_descent` не меняем.**

In [ ]:
def huber_loss(beta, X, y, delta=2.0):
    """
    Возвращает среднюю функцию потерь Хубера для линейной модели.

    Parameters
    ----------
    beta : array-like, shape (q,)
        Коэффициенты линейной модели.
    X : np.ndarray, shape (n, q)
        Матрица признаков со столбцом единиц.
    y : np.ndarray, shape (n,)
        Наблюдаемые значения целевой переменной.
    delta : float
        Порог перехода от квадратичного режима к линейному.

    Returns
    -------
    float
        Среднее значение функции Хубера по объектам.
    """
    residuals = y - X @ beta
    abs_residuals = np.abs(residuals)
    per_object = np.where(
        abs_residuals <= delta,
        0.5 * residuals ** 2,
        delta * (abs_residuals - 0.5 * delta),
    )
    return float(np.mean(per_object))

In [ ]:
def huber_gradient(beta, X, y, delta=2.0):
    """
    Вычисляет градиент среднего критерия Хубера по коэффициентам beta.

    Parameters
    ----------
    beta : array-like, shape (q,)
        Текущие коэффициенты.
    X : np.ndarray, shape (n, q)
        Матрица признаков со столбцом единиц.
    y : np.ndarray, shape (n,)
        Наблюдаемые значения целевой переменной.
    delta : float
        Порог функции Хубера.

    Returns
    -------
    np.ndarray, shape (q,)
        Градиент по beta.
    """
    beta = np.array(beta, dtype=float)

    # TODO:
    # 1) вычислите residuals;
    # 2) ограничьте их через np.clip;
    # 3) примените матричную формулу из markdown выше.
    return ...

In [ ]:
# Проверка на двух объектах.
X_huber_check = np.array([
    [1.0, 0.0],
    [1.0, 1.0],
])
y_huber_check = np.array([0.0, 3.0])
beta_huber_check = np.array([0.0, 0.0])

# При delta=1 residuals = [0, 3], clip -> [0, 1].
# Поэтому градиент равен -(1 / 2) * [1, 1] = [-0.5, -0.5].
assert np.allclose(
    huber_gradient(beta_huber_check, X_huber_check, y_huber_check, delta=1.0),
    [-0.5, -0.5],
)
print("Проверка huber_gradient пройдена.")

In [ ]:
delta = 2.0

beta_huber, huber_history = gradient_descent(
    theta0=np.zeros(2),
    objective_fn=huber_loss,
    grad_fn=huber_gradient,
    learning_rate=0.08,
    n_steps=1200,
    fixed_args=(X, y_contaminated, delta),
)

print("Коэффициенты после обучения по функции Хубера:", beta_huber)

In [ ]:
beta_clean_reference = np.linalg.lstsq(X, y, rcond=None)[0]

plt.figure(figsize=(8.0, 4.8))
plt.scatter(x, y_contaminated, s=27, alpha=0.75, label="загрязнённые данные")
plt.scatter(
    x[outlier_idx],
    y_contaminated[outlier_idx],
    s=85,
    marker="x",
    linewidths=2.0,
    label="изменённые точки",
)
plt.plot(xx, XX @ beta_clean_reference, linewidth=2.0, linestyle="--", label="lstsq на исходных данных")
plt.plot(xx, XX @ beta_mse_contaminated, linewidth=2.0, label="обучение по MSE")
plt.plot(xx, XX @ beta_huber, linewidth=2.0, label="обучение по функции Хубера")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Одинаковые модель и оптимизатор, разные функции потерь")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

### Сравниваем модели по нескольким метрикам

Все метрики ниже считаются **на тех же загрязнённых обучающих данных**, на которых мы только что подбирали параметры. Отложенные валидационные и тестовые выборки сегодня не рассматриваем: цель блока — понять, как две функции потерь ведут себя на одном контролируемом наборе.

Посчитаем

$$
\mathrm{MSE},
\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}},
\qquad
\mathrm{MAE},
\qquad
\mathrm{MAPE}.
$$

RMSE измеряется в тех же единицах, что и целевая переменная. Поскольку квадратный корень — возрастающая функция, на одной и той же выборке MSE и RMSE одинаково упорядочивают модели.

MAPE будем показывать в процентах:

$$
\mathrm{MAPE}
=
\frac1n\sum_i\left|\frac{y_i-\hat y_i}{y_i}\right|\cdot100\%.
$$

В нашем учебном наборе $y_i$ положительны и находятся далеко от нуля. В реальной задаче около $y=0$ MAPE требует особой осторожности.

In [ ]:
def regression_metrics(beta, X, y):
    """Возвращает MSE, RMSE, MAE и MAPE (%) для фиксированных коэффициентов."""
    predictions = X @ beta
    residuals = y - predictions
    mse = np.mean(residuals ** 2)
    return {
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "MAE": np.mean(np.abs(residuals)),
        "MAPE, %": 100.0 * np.mean(np.abs(residuals / y)),
    }


rows = []
for name, beta in [
    ("обучение по MSE", beta_mse_contaminated),
    ("обучение по функции Хубера", beta_huber),
]:
    metrics = regression_metrics(beta, X, y_contaminated)
    rows.append(
        {
            "вариант": name,
            **metrics,
            "beta_0": beta[0],
            "beta_1": beta[1],
            "расстояние до lstsq на исходных данных": np.linalg.norm(beta - beta_clean_reference),
        }
    )

metrics_table = pd.DataFrame(rows).set_index("вариант")
display(metrics_table.round(4))

Обсудите результаты:

1. Какая прямая сильнее изменила наклон из-за трёх крупных отклонений?
2. Почему вариант, обученный по MSE, закономерно имеет меньшие MSE и RMSE **на этой обучающей выборке**?
3. Почему функция Хубера может одновременно иметь немного большие MSE/RMSE, но меньшие MAE и MAPE?
4. Что в этом эксперименте было одинаковым, а что мы действительно изменили?

### Дополнение: почему для MAE нужен другой вариант метода первого порядка

Для одного остатка

$$
L(r)=|r|,
$$

и при $r\ne0$

$$
L'(r)=
\begin{cases}
-1,&r<0,\\
1,&r>0.
\end{cases}
$$

Но в точке $r=0$ обычной производной нет. Поэтому наш сегодняшний контракт `grad_fn`, который предполагает обычный градиент целевой функции, нельзя буквально применить к MAE без дополнительной оговорки.

Для выпуклых негладких функций используют **субградиенты** и субградиентные методы. Например, в нуле для $|r|$ допустим любой субградиент из интервала $[-1,1]$. Подробно такие методы сегодня не реализуем.

Функция Хубера позволяет исследовать робастное поведение, оставаясь в рамках обычного градиентного спуска для дифференцируемой функции.

## 6. Готовая демонстрация: масштаб признаков и геометрия MSE

Этот блок полностью готов: дописывать код не нужно.

На лекции мы обсуждали, что сильно разные масштабы признаков могут сделать целевую функцию вытянутой в пространстве коэффициентов. Здесь посмотрим на это **не на абстрактной квадратичной функции, а прямо на MSE линейной регрессии**.

Чтобы поверхность можно было показать в трёх измерениях, возьмём модель с двумя коэффициентами:

$$
\hat y=\beta_1x_1+\beta_2x_2.
$$

В этой демонстрации признаки и шум центрированы, а $y$ сгенерирован без свободного члена. Поэтому истинный свободный член равен нулю, и для двумерной визуализации пространства коэффициентов достаточно $\beta_1$ и $\beta_2$. Первый признак имеет масштаб порядка единицы, второй — примерно в несколько десятков раз больше.

Сначала построим MSE как функцию $(\beta_1,\beta_2)$ до и после стандартизации. Затем запустим один и тот же градиентный спуск с одним и тем же темпом обучения.

> **Оговорка о стандартизации.** Здесь нет train/test-разделения, потому что мы исследуем только геометрию одной фиксированной задачи оптимизации, а не качество модели на новых данных. В реальной ML-задаче параметры стандартизации оценивают только на обучающей выборке, а затем применяют к валидационным, тестовым и новым данным.

> Этот блок можно пройти самостоятельно, даже если на занятии на него не хватило времени: весь код уже готов.

In [ ]:
rng_scale = np.random.default_rng(7)
n_scale = 80

x1 = rng_scale.normal(0.0, 1.0, size=n_scale)
x2 = rng_scale.normal(0.0, 30.0, size=n_scale)

# Центрируем признаки, чтобы в этой демонстрации обойтись двумя коэффициентами.
x1 = x1 - x1.mean()
x2 = x2 - x2.mean()

noise_scale = rng_scale.normal(0.0, 1.0, size=n_scale)
noise_scale = noise_scale - noise_scale.mean()

y_scale = 2.0 * x1 + 0.05 * x2 + noise_scale

X_raw = np.column_stack([x1, x2])

x1_standardized = x1 / x1.std()
x2_standardized = x2 / x2.std()
X_standardized = np.column_stack([x1_standardized, x2_standardized])

beta_raw_ref = np.linalg.lstsq(X_raw, y_scale, rcond=None)[0]
beta_std_ref = np.linalg.lstsq(X_standardized, y_scale, rcond=None)[0]

print("Стандартные отклонения исходных признаков:")
print(f"x1: {x1.std():.3f}")
print(f"x2: {x2.std():.3f}")

print("\nПосле стандартизации:")
print(f"x1: {x1_standardized.std():.3f}")
print(f"x2: {x2_standardized.std():.3f}")

print("\nМинимум MSE в соответствующих координатах:")
print("исходный масштаб:", beta_raw_ref)
print("после стандартизации:", beta_std_ref)

### Одна и та же задача как поверхность и как карта линий уровня

> Коэффициенты до и после стандартизации нельзя напрямую сравнивать по численным значениям: признаки выражены в разных координатах и масштабах. Это одна и та же линейная зависимость, записанная в разных системах координат параметров. Здесь сравниваем геометрию задачи и сходимость, а не численные значения коэффициентов.

Для каждого набора признаков вычислим MSE на сетке значений $(\beta_1,\beta_2)$.

На объёмном графике высота — это значение MSE. На карте линий уровня те же значения показаны сверху. Минимум находится в самой низкой точке поверхности и в центре вложенных линий уровня.

In [ ]:
def mse_grid(X, y, beta_center, radius=2.5, grid_size=150):
    b1 = np.linspace(beta_center[0] - radius, beta_center[0] + radius, grid_size)
    b2 = np.linspace(beta_center[1] - radius, beta_center[1] + radius, grid_size)
    B1, B2 = np.meshgrid(b1, b2)

    Z = np.empty_like(B1)
    for row in range(grid_size):
        beta = np.column_stack([B1[row], B2[row]])
        predictions = X @ beta.T
        residuals = y[:, None] - predictions
        Z[row] = np.mean(residuals ** 2, axis=0)

    return B1, B2, Z


B1_raw, B2_raw, Z_raw = mse_grid(X_raw, y_scale, beta_raw_ref)
B1_std, B2_std, Z_std = mse_grid(X_standardized, y_scale, beta_std_ref)

fig = plt.figure(figsize=(13.5, 9.5))

ax1 = fig.add_subplot(2, 2, 1, projection="3d")
ax1.plot_surface(B1_raw, B2_raw, Z_raw, alpha=0.8, linewidth=0)
ax1.scatter(
    beta_raw_ref[0],
    beta_raw_ref[1],
    mse_loss(beta_raw_ref, X_raw, y_scale),
    marker="*",
    s=90,
)
ax1.set_xlabel(r"$\beta_1$")
ax1.set_ylabel(r"$\beta_2$")
ax1.set_zlabel("MSE")
ax1.set_title("Исходные масштабы: поверхность MSE")

ax2 = fig.add_subplot(2, 2, 2, projection="3d")
ax2.plot_surface(B1_std, B2_std, Z_std, alpha=0.8, linewidth=0)
ax2.scatter(
    beta_std_ref[0],
    beta_std_ref[1],
    mse_loss(beta_std_ref, X_standardized, y_scale),
    marker="*",
    s=90,
)
ax2.set_xlabel(r"$\beta_1$")
ax2.set_ylabel(r"$\beta_2$")
ax2.set_zlabel("MSE")
ax2.set_title("После стандартизации: поверхность MSE")

ax3 = fig.add_subplot(2, 2, 3)
ax3.contour(B1_raw, B2_raw, Z_raw, levels=18)
ax3.scatter(beta_raw_ref[0], beta_raw_ref[1], marker="*", s=90, label="минимум")
ax3.set_xlabel(r"$\beta_1$")
ax3.set_ylabel(r"$\beta_2$")
ax3.set_title("Исходные масштабы: линии уровня MSE")
ax3.legend()
ax3.grid(alpha=0.15)

ax4 = fig.add_subplot(2, 2, 4)
ax4.contour(B1_std, B2_std, Z_std, levels=18)
ax4.scatter(beta_std_ref[0], beta_std_ref[1], marker="*", s=90, label="минимум")
ax4.set_xlabel(r"$\beta_1$")
ax4.set_ylabel(r"$\beta_2$")
ax4.set_title("После стандартизации: линии уровня MSE")
ax4.legend()
ax4.grid(alpha=0.15)

plt.tight_layout()
plt.show()

До стандартизации небольшое изменение коэффициента при втором признаке гораздо сильнее меняет прогнозы и MSE, потому что сам $x_2$ имеет намного больший масштаб. Поэтому поверхность очень крутая по одной координате и пологая по другой; на карте линий уровня это выглядит как узкая вытянутая долина.

После стандартизации масштабы признаков становятся сопоставимыми. Кривизна MSE по двум координатам тоже становится более сопоставимой, а линии уровня — заметно менее вытянутыми.

Это тот же эффект, который мы обсуждали на лекции через формулу

$$
R(\hat\beta+h)-R(\hat\beta)
=
\frac1n\|Xh\|_2^2.
$$

Если один столбец $X$ намного крупнее другого, одинаковые изменения соответствующих коэффициентов по-разному сильно меняют прогнозы $Xh$ и значение MSE.

In [ ]:
eta_scale_demo = 0.01
n_scale_steps = 8

_, raw_history = gradient_descent(
    theta0=np.zeros(X_raw.shape[1]),
    objective_fn=mse_loss,
    grad_fn=mse_gradient,
    learning_rate=eta_scale_demo,
    n_steps=n_scale_steps,
    fixed_args=(X_raw, y_scale),
)

_, standardized_history = gradient_descent(
    theta0=np.zeros(X_standardized.shape[1]),
    objective_fn=mse_loss,
    grad_fn=mse_gradient,
    learning_rate=eta_scale_demo,
    n_steps=n_scale_steps,
    fixed_args=(X_standardized, y_scale),
)

plt.figure(figsize=(7.5, 4.5))
plt.semilogy(raw_history["objective"], marker="o", label="исходный масштаб")
plt.semilogy(
    standardized_history["objective"],
    marker="o",
    label="после стандартизации",
)
plt.xlabel("Номер шага")
plt.ylabel("MSE, логарифмическая шкала")
plt.title(f"Один и тот же темп обучения: η = {eta_scale_demo}")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

print("MSE в стартовой точке:", raw_history["objective"][0])
print("MSE после шагов, исходный масштаб:", raw_history["objective"][-1])
print("MSE после шагов, стандартизованные признаки:", standardized_history["objective"][-1])

С одним и тем же $\eta$ поведение получилось принципиально разным. В исходных координатах шаг слишком велик для крутого направления, поэтому градиентный спуск расходится. После стандартизации тот же размер шага даёт устойчивое уменьшение MSE.

Масштабирование не добавляет в данные новую информацию и не устраняет мультиколлинеарность. В обычной линейной регрессии без регуляризации `lstsq` может решить исходную задачу напрямую. Здесь нас интересует именно **вычислительный эффект масштаба на геометрию MSE и поведение градиентного спуска**.